# 03 — Explanation audit: faithfulness, stability, localization

**Project:** Quantitatively Audited Explainable AI for Brain Tumor MRI Classification

Notebook 2 produced Grad-CAM and Integrated Gradients maps that *look* reasonable. This notebook
asks whether they deserve to be **trusted**, along three independent axes:

| axis | question | needs the model? |
|---|---|---|
| **faithfulness** | if I remove the pixels the map calls important, does the prediction actually collapse? | yes, live forward passes |
| **stability** | does a tiny, label-preserving change to the input (±5° rotation, +10% brightness) leave the map essentially unchanged? | yes, the maps are recomputed |
| **localization** | does the map land on the tumor a radiologist outlined? | no, saved maps + masks |

These axes can disagree, and those disagreements are the findings. An explainer can localise
nicely (it points at the tumor) without being faithful (the model's decision doesn't depend on
those pixels), or be faithful but so unstable that two near-identical scans get different
explanations. A heatmap's credibility should rest on all three, not on how it looks.

Every metric runs alongside a **random-map control**. Without it, a deletion AUC of 0.3 or an IoU
of 0.25 can't be interpreted: the question is always "better than chance by how much?"

**Inputs to attach** (Add Input → Your Work → Notebooks):

| input | used for | on `sys.path`? |
|---|---|---|
| notebook 0 output | `manifest.csv` + `.mat` files (MD5-matched) | no |
| notebook 1 output | `results/checkpoints/best_model.pt` | no |
| notebook 2 output | `results/explanations/*` + `src/` (incl. `src/explain/`) | **yes, its `src/` only** (copied, see §2b) |

**Outputs** (`/kaggle/working/results/audit/`): `audit_results.csv` (one row per test image ×
method), `audit_summary.json`, `audit_curves.npz` (every deletion/insertion curve),
`audit_config.json`, and the figures in `results/figures/`. `src/` (with the new `src/audit/`) and
`tests/` are saved too.

## 1. GPU check and dependencies

Stability alone recomputes Integrated Gradients (50 steps) three times per test image. On CPU
that would take hours, so the notebook stops here if there's no GPU.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Set Session options -> Accelerator -> GPU, then restart the session.")
DEVICE = torch.device("cuda")
print("device:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
!pip install -q captum

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import skimage
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

WORK = Path("/kaggle/working")
AUDIT_DIR = WORK / "results" / "audit"
FIG_DIR = WORK / "results" / "figures"
for d in [AUDIT_DIR, FIG_DIR, WORK / "tests"]:
    d.mkdir(parents=True, exist_ok=True)
print("scipy", scipy.__version__, "| skimage", skimage.__version__)

## 2. Locate inputs and check they belong together

Three inputs that came from three separate commits could easily be out of sync, e.g. explanations
from an older checkpoint, or a manifest from a re-run of notebook 0 with a different split. Each
link is checked, and the notebook stops if any doesn't match:

1. **Exactly one** `results/checkpoints/best_model.pt` (notebook 1) and **exactly one**
   `results/explanations/test_explanations.npz` (notebook 2) under `/kaggle/input`, at any depth.
2. The checkpoint file's MD5 equals `checkpoint_md5` in notebook 2's `meta.json`, so the maps
   were computed **from this model**.
3. The manifest is picked by the MD5 stored in the checkpoint, so the test split is the one the
   model was evaluated on.
4. The `.npz` row order equals `test_predictions.csv`, which equals the test dataset order.

Nothing is retrained or recomputed to work around a mismatch.

In [ ]:
def find_unique(pattern, parent_names, what, how_to_fix):
    hits = [h for h in sorted(Path("/kaggle/input").rglob(pattern))
            if [p.name for p in h.parents[:len(parent_names)]] == parent_names]
    if len(hits) == 0:
        print("mounted under /kaggle/input:")
        for p in sorted(Path("/kaggle/input").glob("*/*"))[:40]:
            print("   ", p)
        raise FileNotFoundError(f"{what} not found under /kaggle/input. {how_to_fix}")
    if len(hits) > 1:
        raise RuntimeError(f"several copies of {what} attached, keep only one: {hits}")
    return hits[0]


CKPT_PATH = find_unique("best_model.pt", ["checkpoints", "results"], "results/checkpoints/best_model.pt",
                        "Attach notebook 1's committed output. Not retraining here.")
NPZ_PATH = find_unique("test_explanations.npz", ["explanations", "results"], "results/explanations/test_explanations.npz",
                       "Attach notebook 2's committed output. Not recomputing explanations here.")
NB1_DIR, NB2_DIR = CKPT_PATH.parents[2], NPZ_PATH.parents[2]
for required in ["src/explain/gradcam.py", "src/explain/integrated_gradients.py", "src/models/classifier.py",
                 "results/explanations/meta.json", "results/explanations/test_predictions.csv"]:
    if not (NB2_DIR / required).exists():
        raise FileNotFoundError(f"notebook 2 output at {NB2_DIR} is missing {required} -- re-commit notebook 2")

with open(NB2_DIR / "results/explanations/meta.json") as f:
    EXPL_META = json.load(f)
CKPT_MD5 = hashlib.md5(CKPT_PATH.read_bytes()).hexdigest()
if CKPT_MD5 != EXPL_META["checkpoint_md5"]:
    raise RuntimeError(f"explanations were computed from a different checkpoint "
                       f"({EXPL_META['checkpoint_md5']}) than the one attached ({CKPT_MD5}). "
                       "Re-run notebook 2 on this checkpoint, or attach the matching notebook-1 version.")
print("checkpoint :", CKPT_PATH)
print("explanations:", NPZ_PATH, "(computed from this checkpoint)")

### 2b. Code: notebook 2's `src/`, copied, plus `src/audit/`

The same pattern as notebook 2. Only notebook 2's `src/` tree is used, which carries notebook 1's
`data/`, `models/`, `utils/` plus `explain/`. It's copied into `/kaggle/working/src` because this
notebook adds `src/audit/` and inputs are read-only. Every `/kaggle/input` path is removed from
`sys.path`, and an assertion confirms that `src` resolves to `/kaggle/working/src`.

In [ ]:
shutil.copytree(NB2_DIR / "src", WORK / "src", dirs_exist_ok=True)
(WORK / "src" / "audit").mkdir(exist_ok=True)
(WORK / "src" / "audit" / "__init__.py").touch()

sys.path = [p for p in sys.path if not p.startswith("/kaggle/input")]
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

import src
assert Path(src.__file__).resolve().parent == (WORK / "src").resolve(), src.__file__
print("src package:", Path(src.__file__).parent)

In [ ]:
ckpt_raw = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
WANT_MD5 = ckpt_raw["manifest_md5"]

MANIFEST_PATH = DATA_ROOT = None
for m in sorted(Path("/kaggle/input").rglob("manifest.csv")):
    if hashlib.md5(m.read_bytes()).hexdigest() != WANT_MD5:
        continue
    first = pd.read_csv(m, nrows=1).file_path.iloc[0]
    for root in [m.parent / "data", m.parent.parent / "data"]:
        if (root / first).exists():
            MANIFEST_PATH, DATA_ROOT = m, root
            break
    if MANIFEST_PATH:
        break
if MANIFEST_PATH is None:
    raise FileNotFoundError(f"no attached manifest.csv with md5 {WANT_MD5} (+ its data/ folder). "
                            "Attach the notebook-0 version notebook 1 was trained on.")
assert EXPL_META["manifest_md5"] == WANT_MD5
print("manifest :", MANIFEST_PATH, "(md5 matches checkpoint and explanations)")
print("data root:", DATA_ROOT)

## 3. Model, test set, saved maps

The model is loaded with notebook 1's `load_checkpoint`: `pretrained=False`, so there's no ImageNet
download; it's in eval mode on the GPU. The explainer settings (Grad-CAM layer, IG steps) come from
notebook 2's `meta.json`, **not** from defaults. When stability recomputes a map on a perturbed
image, it has to use exactly the settings that produced the original map; otherwise part of any
difference would come from changed settings, not from the perturbation.

In [ ]:
from src.utils.seed import set_seed
from src.models.classifier import load_checkpoint
from src.data.dataset import BrainTumorDataset
from src.data.transforms import get_eval_transforms, get_stability_transform, IMAGENET_MEAN, IMAGENET_STD
from src.explain import EXPLAINERS

model, ckpt = load_checkpoint(CKPT_PATH, DEVICE)
assert not model.training
CFG = ckpt["config"]
CLASS_NAMES = ckpt["class_names"]
SEED = CFG["seed"]
IMAGE_SIZE = CFG["image_size"]
set_seed(SEED)

EXPL_KW = {
    "gradcam": {"target_layer": EXPL_META["gradcam"]["target_layer"]},
    "integrated_gradients": {"n_steps": EXPL_META["integrated_gradients"]["n_steps"]},
}
print("explainer settings (from notebook 2):", EXPL_KW)

manifest = pd.read_csv(MANIFEST_PATH)
key = pd.read_csv(NB2_DIR / "results/explanations/test_predictions.csv")
test_ds = BrainTumorDataset(manifest, DATA_ROOT, "test", get_eval_transforms(IMAGE_SIZE))

z = dict(np.load(NPZ_PATH))   # ~200 MB in RAM; random access is much faster than the lazy NpzFile
N = len(key)
assert len(test_ds) == N == len(z["file_path"])
assert (z["file_path"] == key.file_path.to_numpy()).all() and list(test_ds.df.file_path) == list(key.file_path)
assert (z["true"] == key.true.to_numpy()).all()
for k in ["gradcam", "ig", "ig_signed", "mask", "target"]:
    print(f"{k:10s} {str(z[k].shape):18s} {z[k].dtype}")
print(f"\n{N} test images, row order consistent across npz / test_predictions.csv / dataset")

### Audit configuration

Values from the training config (`audit:` section) take precedence; anything missing gets the
default below. The final values are written to `audit_config.json`, so the reported numbers can be
traced to exact settings.

- `subset_n`: `None` = full test set. Set to an integer for a quicker, **seeded, class-balanced**
  sample for faithfulness and stability. Localization always uses all images, since it only reads
  saved arrays.
- `insertion_blur_sigma`: σ in pixels of the Gaussian blur used as the insertion starting image.
  10 px on a 224 image removes the tumor's texture and edges but keeps the coarse brain shape. That
  is what RISE-style insertion wants: the model shouldn't recognise the class from the blurred
  start, while the image is still a plausible input and not an out-of-distribution grey square.

In [ ]:
AUDIT_DEFAULTS = {
    "deletion_insertion_steps": 20,
    "stability_transforms": [{"type": "rotate", "degrees": 5}, {"type": "rotate", "degrees": -5},
                             {"type": "brightness", "factor": 1.10}],
    "localization_threshold": 0.5,
    "subset_n": None,
    "insertion_blur_sigma": 10.0,
    "n_bootstrap": 1000,
}
ACFG = {**AUDIT_DEFAULTS, **CFG.get("audit", {})}
STEPS = ACFG["deletion_insertion_steps"]
SPECS = ACFG["stability_transforms"]
THR = ACFG["localization_threshold"]

with open(AUDIT_DIR / "audit_config.json", "w") as f:
    json.dump({"audit": ACFG, "explainers": EXPL_KW, "checkpoint_md5": CKPT_MD5, "manifest_md5": WANT_MD5}, f, indent=2)
print(json.dumps(ACFG, indent=2))

## 4. The metrics, in `src/audit/`

Every metric is a plain function on arrays or tensors, with no notebook globals, so it can be
unit-tested on inputs where the right answer is known (§5). That matters for this project: the
claim is "these explainers are or aren't trustworthy", and that claim can only be as reliable as
the code that measures it.

### 4a. Faithfulness: deletion and insertion (Petsiuk et al., RISE, 2018)

Rank all 224×224 pixels by the heatmap, then perturb them in `steps` equal chunks, most important
first, recording the probability of the **originally predicted** class after each chunk:

- **Deletion:** start from the real image and *replace* the top-ranked pixels with the baseline,
  black in pixel space (= `−mean/std` after normalisation, the same baseline IG uses). If the map
  is faithful, the probability collapses early. **Lower AUC = more faithful.**
- **Insertion:** start from a heavily blurred image and *restore* the top-ranked pixels to their
  real values. If the map is faithful, the probability recovers early. **Higher AUC = more faithful.**

Details that change the numbers, and the choices made:

- **Ties are broken by a seeded random permutation.** Grad-CAM maps have large areas exactly at 0,
  and IG's saved maps are clipped at 1 for the top 1%. Breaking ties by raster order (what `argsort`
  does) would always delete the top rows of the image first, a systematic bias that has nothing to
  do with the explanation.
- **IG is ranked by its unclipped magnitude** `|ig_signed|`, not by the clipped [0, 1] map.
  Ranking only depends on order, and clipping turns the top 1% into a tie, which throws away
  exactly the information deletion tests.
- All `steps + 1` perturbed images for one curve go through the model in **one batch**: 21 images,
  a single forward pass.
- AUC is the trapezoid rule on an x-axis normalised to [0, 1] (the fraction of pixels perturbed),
  so AUCs lie in [0, 1] and can be compared across step counts.

In [ ]:
%%writefile /kaggle/working/src/audit/faithfulness.py
import math

import numpy as np
import torch
import torchvision.transforms.functional as TF

from src.data.transforms import IMAGENET_MEAN, IMAGENET_STD


def pixel_order(heatmap, seed=0):
    # most important first; seeded tie-break so flat regions aren't removed in raster order
    h = np.asarray(heatmap, dtype=np.float64).ravel()
    tie = np.random.default_rng(seed).random(h.size)
    return np.lexsort((tie, -h))


def step_masks(order, steps):
    # (steps+1, n_pixels) bool; row k marks the pixels perturbed after k of `steps` chunks
    n = order.size
    rank = np.empty(n, dtype=np.int64)
    rank[order] = np.arange(n)
    counts = np.round(np.linspace(0, n, steps + 1)).astype(np.int64)
    return rank[None, :] < counts[:, None]


def black_baseline(image):
    mean = torch.tensor(IMAGENET_MEAN, device=image.device).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=image.device).view(3, 1, 1)
    return ((0.0 - mean) / std).expand_as(image).contiguous()


def blurred_baseline(image, sigma=10.0):
    k = 2 * math.ceil(3 * sigma) + 1
    return TF.gaussian_blur(image.unsqueeze(0), kernel_size=[k, k], sigma=[sigma, sigma])[0]


@torch.no_grad()
def perturbation_curve(model, image, heatmap, target, mode, steps=20, start=None, seed=0, batch_size=64):
    # image: normalised (C, H, W). deletion: image -> start (baseline); insertion: start (blurred) -> image.
    assert not model.training
    device = next(model.parameters()).device
    image = image.to(device)
    C, H, W = image.shape
    if start is None:
        start = black_baseline(image) if mode == "deletion" else blurred_baseline(image)
    start = start.to(device)
    masks = torch.from_numpy(step_masks(pixel_order(heatmap, seed), steps)).to(device).view(steps + 1, 1, H, W)
    if mode == "deletion":
        batch = torch.where(masks, start[None], image[None])
    elif mode == "insertion":
        batch = torch.where(masks, image[None], start[None])
    else:
        raise ValueError(mode)
    probs = [model(batch[i:i + batch_size]).float().softmax(1)[:, int(target)] for i in range(0, len(batch), batch_size)]
    return torch.cat(probs).cpu().numpy()


def auc(curve):
    # trapezoid rule on x in [0, 1] with equal spacing
    c = np.asarray(curve, dtype=np.float64)
    return float(((c[:-1] + c[1:]) / 2).mean())

### 4b. Stability

For each perturbation in `stability_transforms`, the perturbed image is produced by **notebook 1's
own** `get_stability_transform` (resize → rotate/brighten on uint8 → normalise), so it's the same
pipeline the model was trained and evaluated with. The map is recomputed with the same explainer,
the same settings and the same **target class** as the original. Holding the target fixed means
the comparison is "same question, slightly different image", even when the perturbation flips the
prediction (those cases are counted separately as `pred_changed`).

**Alignment, the key correctness point.** After a +5° rotation the tumor has moved, so a
*perfectly stable* explainer's map moves with it. Correlating that rotated map against the
unrotated original would penalise the explainer for being correct. So the perturbed map is first
**rotated back by −5°** (bilinear, as the map is smooth), and only then compared.

Rotation also brings in black corners that have no counterpart in the original, and rotating
back moves them again. Those pixels are excluded via a **validity mask**: a ones-image is sent
through the forward rotation (nearest-neighbour, like the image) and then the inverse
(bilinear), and only pixels that come back as ~1 are kept. Brightness is a pixel-wise change, so
it needs no alignment and every pixel is valid.

Two similarity measures, because they fail differently:
- **Spearman rank correlation** over valid pixels: does the map keep the same *ordering* of
  importance? It ignores the scale, which matters given the per-image normalisation.
- **SSIM** (structural similarity, 7×7 window, data range 1) averaged over valid pixels: is the
  local *structure* preserved, i.e. the same blobs and edges in the same places? Speckly IG maps
  can keep a decent global rank correlation while their local structure changes completely; SSIM
  catches that.

**Caveat for sparse maps:** both measures give credit where *both* maps are ~0. SSIM is ≈1 on
shared empty background, and Spearman counts tied zeros as agreeing. A Grad-CAM map that is zero
over most of the brain therefore gets a boost from that empty area alone. Keep this in mind when
comparing Grad-CAM's stability with IG's, which is non-zero almost everywhere. The unit tests below
use a pattern with structure everywhere, so this effect can't hide a misalignment.

Both return `NaN` for a constant map (e.g. an all-zero Grad-CAM), because neither is defined
there. They aren't silently set to 0 or 1.

In [ ]:
%%writefile /kaggle/working/src/audit/stability.py
import numpy as np
import torch
from scipy.stats import spearmanr
from skimage.metrics import structural_similarity
from torchvision.transforms import InterpolationMode
from torchvision.transforms.v2 import functional as F


def spec_name(spec):
    if spec["type"] == "rotate":
        return f"rotate{spec['degrees']:+g}"
    return f"brightness_x{spec['factor']:g}"


def _rotate(arr, degrees, interpolation):
    t = torch.as_tensor(np.asarray(arr, dtype=np.float32))[None]
    return F.rotate(t, angle=float(degrees), interpolation=interpolation, fill=0.0)[0].numpy()


def align_to_original(heatmap, spec):
    # undo the geometric part of the perturbation so both maps live in the original image frame
    if spec["type"] == "rotate":
        return _rotate(heatmap, -spec["degrees"], InterpolationMode.BILINEAR)
    return np.asarray(heatmap, dtype=np.float32)


def valid_region(shape, spec):
    if spec["type"] != "rotate":
        return np.ones(shape, dtype=bool)
    # nearest forward (as the image pipeline does), bilinear back (as align_to_original does)
    fwd = _rotate(np.ones(shape, np.float32), spec["degrees"], InterpolationMode.NEAREST)
    return _rotate(fwd, -spec["degrees"], InterpolationMode.BILINEAR) > 0.999


def spearman_similarity(a, b, valid=None):
    a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)
    if valid is not None:
        a, b = a[valid], b[valid]
    a, b = a.ravel(), b.ravel()
    if a.std() == 0 or b.std() == 0:
        return float("nan")
    return float(spearmanr(a, b)[0])


def ssim_similarity(a, b, valid=None):
    a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)
    if a.std() == 0 or b.std() == 0:
        return float("nan")
    _, smap = structural_similarity(a, b, data_range=1.0, full=True)
    return float(smap[valid].mean() if valid is not None else smap.mean())

### 4c. Localization

All four metrics compare one heatmap with one binary tumor mask (both 224×224):

- **IoU and Dice** of the binarised map (`heat ≥ localization_threshold`, 0.5) against the mask.
  These depend on the threshold *and* on the normalisation. IG's p99 normalisation from notebook 2
  was chosen partly so that 0.5 isn't meaningless for IG.
- **Pointing game** (Zhang et al., 2016): is the map's peak inside the mask? If several pixels tie
  for the max, it's a hit when the majority of them fall inside. For IG the peak is taken on the
  unclipped `|ig_signed|`; on the clipped map the top 1% all tie at exactly 1.0.
- **Energy in mask:** the fraction of total heatmap mass inside the mask. Unlike IoU it needs no
  threshold, so it's the most robust of the four. **Chance level equals the mask's area fraction**,
  so I also report `energy_ratio = energy / mask_frac` (1 = chance, >1 = concentrated on the tumor).

**Tiny or empty masks.** An empty mask (possible after resizing for a very small tumor) makes
every metric undefined, so it returns `NaN` and gets counted, not dropped silently. Tiny masks are
kept but the area fraction `mask_frac` is stored per row, so the analysis can stratify by tumor
size. A 7×7 Grad-CAM is structurally unable to get a high IoU on a tumor smaller than one of its
cells, and that's a fact about resolution, not about faithfulness.

In [ ]:
%%writefile /kaggle/working/src/audit/localization.py
import numpy as np


def _mask(mask):
    return np.asarray(mask).astype(bool)


def iou_dice(heatmap, mask, threshold=0.5):
    m = _mask(mask)
    if not m.any():
        return float("nan"), float("nan")
    b = np.asarray(heatmap) >= threshold
    inter = np.logical_and(b, m).sum()
    iou = inter / np.logical_or(b, m).sum()
    dice = 2 * inter / (b.sum() + m.sum())
    return float(iou), float(dice)


def pointing_game(heatmap, mask):
    m = _mask(mask)
    if not m.any():
        return float("nan")
    h = np.asarray(heatmap)
    peak = h == h.max()
    # ties (clipped or flat maps) count as a hit only if most of the tied peak is inside
    return float(m[peak].mean() >= 0.5)


def energy_in_mask(heatmap, mask):
    m = _mask(mask)
    h = np.asarray(heatmap, dtype=np.float64)
    assert h.min() >= 0, "energy needs a non-negative map"
    total = h.sum()
    if not m.any() or total <= 0:
        return float("nan")
    return float(h[m].sum() / total)


def mask_fraction(mask):
    return float(_mask(mask).mean())

In [ ]:
%%writefile /kaggle/working/src/audit/__init__.py
from src.audit import faithfulness, localization, stability

## 5. Sanity tests for the metric code

These run **before** the expensive loop, and the notebook stops if any fails. Each test sets up a
situation where the correct answer is known by construction:

- **Localization:** a map equal to the mask scores IoU = Dice = 1, a pointing hit and energy 1; the
  inverted map scores 0 everywhere. A uniform random map gets energy ≈ mask area and pointing hit
  rate ≈ mask area, i.e. chance. Empty masks give `NaN`.
- **Faithfulness:** a toy "model" whose logit depends only on one region of the image. A heatmap
  that marks exactly that region has to give a *lower* deletion AUC and a *higher* insertion AUC
  than a random heatmap. If it doesn't, the curve code has a direction or sign bug. Also: AUC of
  known curves, step-mask monotonicity, and deterministic tie-breaking.
- **Stability:** identical maps give similarity 1. Rotating a map with **the same
  `v2.RandomRotation((5, 5))` that `get_stability_transform` uses** and then aligning it back
  recovers the original (Spearman > 0.98), while the *unaligned* comparison scores lower. This test
  pins the rotation direction: if `align_to_original` rotated the wrong way, it would fail.

They're written to `tests/test_audit_metrics.py` and saved with the output, so they can also be run
outside this notebook (`python -m pytest tests/` from the output folder).

In [ ]:
%%writefile /kaggle/working/tests/test_audit_metrics.py
import numpy as np
import pytest
import torch
from torch import nn
from torchvision.transforms import v2

from src.audit import faithfulness as F
from src.audit import localization as L
from src.audit import stability as S


def square_mask(n=224, lo=80, hi=150):
    m = np.zeros((n, n), np.uint8)
    m[lo:hi, lo:hi] = 1
    return m


# ---------------- localization ----------------

def test_perfect_map_localizes():
    m = square_mask()
    h = m.astype(np.float32)
    assert L.iou_dice(h, m) == (1.0, 1.0)
    assert L.pointing_game(h, m) == 1.0
    assert L.energy_in_mask(h, m) == pytest.approx(1.0)


def test_inverted_map_scores_zero():
    m = square_mask()
    h = 1.0 - m.astype(np.float32)
    assert L.iou_dice(h, m) == (0.0, 0.0)
    assert L.pointing_game(h, m) == 0.0
    assert L.energy_in_mask(h, m) == pytest.approx(0.0)


def test_random_map_is_at_chance():
    m = square_mask()
    frac = L.mask_fraction(m)
    rng = np.random.default_rng(0)
    energies, hits = [], []
    for _ in range(300):
        h = rng.random(m.shape)
        energies.append(L.energy_in_mask(h, m))
        hits.append(L.pointing_game(h, m))
    assert np.mean(energies) == pytest.approx(frac, abs=0.01)
    assert np.mean(hits) == pytest.approx(frac, abs=0.06)
    iou, _ = L.iou_dice(rng.random(m.shape), m)
    assert iou < 0.2


def test_empty_mask_is_nan():
    m = np.zeros((32, 32), np.uint8)
    h = np.random.default_rng(0).random((32, 32))
    assert all(np.isnan(v) for v in [*L.iou_dice(h, m), L.pointing_game(h, m), L.energy_in_mask(h, m)])


def test_tied_peak_majority_rule():
    m = np.zeros((10, 10), np.uint8); m[:, :3] = 1
    flat = np.ones((10, 10))                       # every pixel is the peak, 30% inside -> miss
    assert L.pointing_game(flat, m) == 0.0
    h = np.zeros((10, 10)); h[:, :2] = 1; h[:, 9] = 1   # 20 of 30 tied peak pixels inside -> hit
    assert L.pointing_game(h, m) == 1.0


# ---------------- faithfulness ----------------

class RegionModel(nn.Module):
    # logits [s, -s] with s = 2 * (mean region intensity - 0.5): p(class 0) goes 0.02 -> 0.98 roughly
    # in proportion to how much of the region is present (a steeper gain saturates and hides differences)
    def __init__(self, region):
        super().__init__()
        self.register_buffer("region", torch.as_tensor(region, dtype=torch.bool))
        self.dummy = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        s = 2 * (x[:, :, self.region].mean(dim=(1, 2)) - 0.5) + self.dummy
        return torch.stack([s, -s], dim=1)


def _toy():
    region = np.zeros((32, 32), bool); region[4:12, 4:12] = True
    model = RegionModel(region).eval()
    image = torch.ones(3, 32, 32)
    zeros = torch.zeros(3, 32, 32)
    return model, image, zeros, region


def test_auc_known_curves():
    assert F.auc(np.ones(21)) == pytest.approx(1.0)
    assert F.auc(np.linspace(1, 0, 21)) == pytest.approx(0.5)


def test_step_masks_monotone():
    order = F.pixel_order(np.random.default_rng(1).random((8, 8)))
    m = F.step_masks(order, steps=8)
    assert m.shape == (9, 64) and not m[0].any() and m[-1].all()
    assert (np.diff(m.sum(1)) > 0).all()
    assert (m[:-1] <= m[1:]).all()          # once perturbed, stays perturbed


def test_tie_break_is_seeded():
    flat = np.zeros((8, 8))
    assert (F.pixel_order(flat, 3) == F.pixel_order(flat, 3)).all()
    assert not (F.pixel_order(flat, 3) == F.pixel_order(flat, 4)).all()
    assert not (F.pixel_order(flat, 3) == np.arange(64)).all()   # not raster order


def test_perfect_explainer_beats_random_deletion_and_insertion():
    model, image, zeros, region = _toy()
    perfect = region.astype(np.float32)
    rand = np.random.default_rng(0).random((32, 32))
    kw = dict(target=0, steps=16, start=zeros)
    del_perfect = F.auc(F.perturbation_curve(model, image, perfect, mode="deletion", **kw))
    del_random = F.auc(F.perturbation_curve(model, image, rand, mode="deletion", **kw))
    ins_perfect = F.auc(F.perturbation_curve(model, image, perfect, mode="insertion", **kw))
    ins_random = F.auc(F.perturbation_curve(model, image, rand, mode="insertion", **kw))
    assert del_perfect < del_random - 0.1
    assert ins_perfect > ins_random + 0.1


def test_curve_endpoints():
    model, image, zeros, region = _toy()
    h = np.random.default_rng(0).random((32, 32))
    d = F.perturbation_curve(model, image, h, target=0, mode="deletion", steps=8, start=zeros)
    i = F.perturbation_curve(model, image, h, target=0, mode="insertion", steps=8, start=zeros)
    full, empty = torch.softmax(model(image[None]), 1)[0, 0].item(), torch.softmax(model(zeros[None]), 1)[0, 0].item()
    assert d[0] == pytest.approx(full) and d[-1] == pytest.approx(empty)
    assert i[0] == pytest.approx(empty) and i[-1] == pytest.approx(full)


# ---------------- stability ----------------

def _blob(n=224, cy=80, cx=140, s=6):
    yy, xx = np.mgrid[:n, :n]
    return np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * s ** 2)).astype(np.float32)


def test_identical_maps_similarity_one():
    h = _blob()
    assert S.spearman_similarity(h, h) == pytest.approx(1.0)
    assert S.ssim_similarity(h, h) == pytest.approx(1.0)


def test_constant_map_is_nan():
    assert np.isnan(S.spearman_similarity(np.zeros((16, 16)), _blob(16, 8, 8, 3)))
    assert np.isnan(S.ssim_similarity(np.zeros((16, 16)), _blob(16, 8, 8, 3)))


def _smooth_field(n=224, sigma=4, seed=0):
    # structure everywhere: a mostly-zero map would let SSIM's background (~1 where both are 0) hide misalignment
    from scipy.ndimage import gaussian_filter
    f = gaussian_filter(np.random.default_rng(seed).random((n, n)), sigma)
    return ((f - f.min()) / (f.max() - f.min())).astype(np.float32)


@pytest.mark.parametrize("deg", [5, -5])
def test_rotation_alignment_matches_pipeline(deg):
    h = _smooth_field()
    spec = {"type": "rotate", "degrees": deg}
    rotated = v2.RandomRotation(degrees=(deg, deg))(torch.from_numpy(h)[None])[0].numpy()  # same op as get_stability_transform
    valid = S.valid_region(h.shape, spec)
    aligned = S.align_to_original(rotated, spec)
    assert 0.9 < valid.mean() < 1.0
    assert S.spearman_similarity(h, aligned, valid) > 0.95
    assert S.ssim_similarity(h, aligned, valid) > 0.9
    assert S.ssim_similarity(h, rotated, valid) < S.ssim_similarity(h, aligned, valid) - 0.2
    assert S.spearman_similarity(h, rotated, valid) < S.spearman_similarity(h, aligned, valid) - 0.2
    wrong_way = S.align_to_original(rotated, {"type": "rotate", "degrees": -deg})
    assert S.ssim_similarity(h, wrong_way, valid) < S.ssim_similarity(h, aligned, valid) - 0.2


def test_brightness_needs_no_alignment():
    h = _blob()
    spec = {"type": "brightness", "factor": 1.1}
    assert (S.align_to_original(h, spec) == h).all()
    assert S.valid_region(h.shape, spec).all()

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_audit_metrics.py"],
                   cwd=WORK, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "metric sanity tests failed -- fix src/audit before trusting any number below"

## 6. Which images get the model-based audit

Localization runs on all `N` test images. Faithfulness and stability run on `audit_idx`, which is
the full test set by default. With `subset_n` set, it's a seeded sample with the same number of
images per **true** class (or all of a class, if it has fewer). That keeps a small run from being
dominated by glioma. The indices are sorted, so outputs keep test-set order.

The **random control** gets its own seeded map per image (`seed = SEED·100003 + row`). It's the
same map in every metric family and in every rerun, so its scores are reproducible too.

In [ ]:
from src.audit import faithfulness as AF, stability as AS, localization as AL

METHODS = ["gradcam", "integrated_gradients", "random"]   # random = control, not an explainer
EXPLAIN_METHODS = ["gradcam", "integrated_gradients"]
H = W = IMAGE_SIZE
true = z["true"]


def random_map(i):
    return np.random.default_rng(SEED * 100003 + i).random((H, W)).astype(np.float32)


def display_map(m, i):
    # the [0,1] map as saved/shown -- used for thresholded and mass-based metrics
    return {"gradcam": lambda: z["gradcam"][i].astype(np.float32),
            "integrated_gradients": lambda: z["ig"][i].astype(np.float32),
            "random": lambda: random_map(i)}[m]()


def rank_map(m, i):
    # order-based metrics (deletion/insertion ranking, pointing peak) use IG's unclipped magnitude
    return np.abs(z["ig_signed"][i]) if m == "integrated_gradients" else display_map(m, i)


subset_n = ACFG["subset_n"]
if subset_n is None or subset_n >= N:
    audit_idx = np.arange(N)
else:
    rng = np.random.default_rng(SEED)
    per_class = subset_n // len(CLASS_NAMES)
    audit_idx = np.sort(np.concatenate([
        rng.choice(np.flatnonzero(true == k), size=min(per_class, (true == k).sum()), replace=False)
        for k in range(len(CLASS_NAMES))]))
print(f"model-based audit on {len(audit_idx)} of {N} images:",
      dict(zip(CLASS_NAMES, np.bincount(true[audit_idx], minlength=len(CLASS_NAMES)).tolist())))

### Reproducibility check before perturbing anything

The stability numbers only mean something if recomputing a map on the *unperturbed* image
reproduces the saved map. Otherwise part of the measured "instability" would be numerical noise
(cuDNN nondeterminism, a different GPU) and not the effect of the perturbation. This recomputes
both maps on a few images and compares them with notebook 2's saved maps. They should be
essentially identical: Spearman ≥ 0.99.

In [ ]:
for i in audit_idx[:3]:
    x = test_ds[int(i)]["image"]
    for m in EXPLAIN_METHODS:
        fresh = EXPLAINERS[m](model, x, target=int(z["target"][i]), **EXPL_KW[m])
        rho = AS.spearman_similarity(fresh, display_map(m, i))
        print(f"row {i:3d} {m:22s} spearman(fresh, saved) = {rho:.4f}")
        assert rho >= 0.99 or np.isnan(rho), "recomputed map differs from notebook 2's -- explainer code or settings changed"

## 7. Faithfulness and stability over the test set

One pass over `audit_idx`. Per image:

- **Faithfulness:** 3 methods × {deletion, insertion}, i.e. 6 batched forward passes of 21 images.
- **Stability:** 3 perturbations, each with one forward pass (for `pred_changed`) plus a Grad-CAM
  recomputation and an IG recomputation. This is the expensive part: IG costs ~50 forward and
  backward passes each time.

The perturbed images come from separate `DataLoader`s over the same `Subset`, each using notebook
1's `get_stability_transform`, so the `.mat` decoding happens in background workers. Each loaded
batch is checked against the expected `file_path`, so a loader that drifted out of step would stop
the run instead of quietly pairing the wrong images.

Timing is estimated after the first 5 images and printed, so you can decide whether to set
`subset_n` before waiting for the full run.

In [ ]:
def loader_for(transform):
    ds = BrainTumorDataset(manifest, DATA_ROOT, "test", transform)
    return DataLoader(Subset(ds, audit_idx.tolist()), batch_size=1, shuffle=False, num_workers=1)


base_loader = loader_for(get_eval_transforms(IMAGE_SIZE))
pert_loaders = [loader_for(get_stability_transform(spec, IMAGE_SIZE)) for spec in SPECS]
SPEC_NAMES = [AS.spec_name(s) for s in SPECS]
VALID = {name: AS.valid_region((H, W), spec) for name, spec in zip(SPEC_NAMES, SPECS)}
print({n: f"{v.mean():.3f} of pixels valid" for n, v in VALID.items()})

curves = {(m, mode): np.full((N, STEPS + 1), np.nan, np.float32) for m in METHODS for mode in ["deletion", "insertion"]}
faith = {}   # (row, method) -> dict
stab = {}    # (row, method) -> dict
pert_pred = {}  # row -> dict

t_start = time.time()
for j, (b, *pbs) in enumerate(tqdm(zip(base_loader, *pert_loaders), total=len(audit_idx), desc="audit")):
    i = int(audit_idx[j])
    assert b["file_path"][0] == key.file_path[i] and all(pb["file_path"][0] == key.file_path[i] for pb in pbs)
    x = b["image"][0].to(DEVICE)
    tgt = int(z["target"][i])
    starts = {"deletion": AF.black_baseline(x), "insertion": AF.blurred_baseline(x, ACFG["insertion_blur_sigma"])}

    for m in METHODS:
        row = {}
        for mode in ["deletion", "insertion"]:
            c = AF.perturbation_curve(model, x, rank_map(m, i), tgt, mode, steps=STEPS, start=starts[mode], seed=i)
            curves[(m, mode)][i] = c
            row[f"{mode}_auc"] = AF.auc(c)
        row["p_target_original"] = float(curves[(m, "deletion")][i][0])
        faith[(i, m)] = row

    pp = {}
    for name, pb in zip(SPEC_NAMES, pbs):
        xp = pb["image"][0].to(DEVICE)
        with torch.no_grad():
            pr = model(xp[None]).softmax(1)[0]
        pp[f"pred_changed_{name}"] = float(int(pr.argmax()) != tgt)
        pp[f"p_target_{name}"] = float(pr[tgt])
        for m in EXPLAIN_METHODS:
            hp = EXPLAINERS[m](model, xp, target=tgt, **EXPL_KW[m])
            aligned = AS.align_to_original(hp, dict(zip(SPEC_NAMES, SPECS))[name])
            orig = display_map(m, i)
            stab.setdefault((i, m), {})[f"spearman_{name}"] = AS.spearman_similarity(orig, aligned, VALID[name])
            stab[(i, m)][f"ssim_{name}"] = AS.ssim_similarity(orig, aligned, VALID[name])
    pert_pred[i] = pp

    if j == 4:
        per_img = (time.time() - t_start) / 5
        print(f"~{per_img:.2f}s/image -> estimated {per_img * len(audit_idx) / 60:.1f} min for {len(audit_idx)} images")

elapsed = time.time() - t_start
print(f"done: {elapsed / 60:.1f} min actual ({elapsed / len(audit_idx):.2f}s/image)")

## 8. Localization over the full test set

This needs no model, only the saved maps and masks, so it covers all `N` images whatever
`subset_n` is set to.

In [ ]:
loc = {}
for i in range(N):
    mask = z["mask"][i]
    frac = AL.mask_fraction(mask)
    for m in METHODS:
        heat = display_map(m, i)
        iou, dice = AL.iou_dice(heat, mask, THR)
        energy = AL.energy_in_mask(heat, mask)
        loc[(i, m)] = {
            "mask_frac": frac, "iou": iou, "dice": dice,
            "pointing_hit": AL.pointing_game(rank_map(m, i), mask),
            "energy_in_mask": energy,
            "energy_ratio": energy / frac if frac > 0 else float("nan"),
            "heat_area_above_thr": float((heat >= THR).mean()),
        }
n_empty = sum(np.isnan(v["iou"]) for (i, m), v in loc.items() if m == "gradcam")
print(f"localization done for {N} images x {len(METHODS)} methods; {n_empty} empty masks (NaN, excluded from means)")

## 9. Assemble `audit_results.csv`

One row per **test image × method**, ordered by test-set row (the same order as
`test_predictions.csv`), then by method. Faithfulness and stability columns are `NaN` for images
outside `audit_idx` (marked by `in_model_audit`) and, for stability, for the random control, which
isn't an explainer and has nothing to recompute. Prediction-level columns like `pred_changed_*`
are repeated on each method's row so every row is self-contained.

In [ ]:
in_audit = set(audit_idx.tolist())
rows = []
for i in range(N):
    base = {"row": i, "file_path": key.file_path[i], "patient_id": key.patient_id[i],
            "true": int(true[i]), "true_name": CLASS_NAMES[true[i]],
            "pred": int(z["pred"][i]), "pred_name": CLASS_NAMES[z["pred"][i]],
            "correct": bool(true[i] == z["pred"][i]), "confidence": float(key.confidence[i]),
            "in_model_audit": i in in_audit}
    for m in METHODS:
        r = {**base, "method": m, **loc[(i, m)]}
        r.update(faith.get((i, m), {}))
        r.update(stab.get((i, m), {}))
        r.update(pert_pred.get(i, {}))
        rows.append(r)

audit = pd.DataFrame(rows)
STAB_COLS = [f"{s}_{n}" for n in SPEC_NAMES for s in ["spearman", "ssim"]]
for c in ["deletion_auc", "insertion_auc", "p_target_original", *STAB_COLS]:
    if c not in audit:
        audit[c] = np.nan
audit.to_csv(AUDIT_DIR / "audit_results.csv", index=False)

np.savez_compressed(AUDIT_DIR / "audit_curves.npz", file_path=key.file_path.to_numpy().astype(str),
                    x=np.linspace(0, 1, STEPS + 1),
                    **{f"{m}__{mode}": c for (m, mode), c in curves.items()})
print(audit.shape)
audit.head(6)

## 10. Summary with patient-level confidence intervals

Each metric gets a mean, median, std and count per method, plus a **95% CI from a patient-cluster
bootstrap**. The test set has ~35 patients with several slices each, and slices from one patient
are strongly correlated. A naive per-image CI would treat ~460 images as ~460 independent pieces of
evidence and come out far too narrow. Resampling *patients*, with all their slices together,
gives an interval that reflects how many independent cases there really are.

The same bootstrap gives CIs for the **paired differences** Grad-CAM − IG, Grad-CAM − random and
IG − random, computed per image and then resampled by patient. If an interval excludes 0, the
difference holds up against patient sampling. If it doesn't, the two methods can't be told apart
on this test set, and the write-up should say exactly that.

In [ ]:
METRICS = ["deletion_auc", "insertion_auc", "iou", "dice", "pointing_hit", "energy_in_mask", "energy_ratio", *STAB_COLS]
HIGHER_IS_BETTER = {"deletion_auc": False}


def cluster_bootstrap_ci(values, groups, n_boot, seed):
    d = pd.DataFrame({"v": values, "g": groups}).dropna()
    if d.empty:
        return [float("nan"), float("nan")]
    g = d.groupby("g")["v"].agg(["sum", "count"])
    s, c = g["sum"].to_numpy(), g["count"].to_numpy()
    idx = np.random.default_rng(seed).integers(0, len(g), size=(n_boot, len(g)))
    means = s[idx].sum(1) / c[idx].sum(1)
    return [float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))]


summary = {"n_test": int(N), "n_model_audit": int(len(audit_idx)), "n_patients_test": int(key.patient_id.nunique()),
           "config": ACFG, "explainers": EXPL_KW, "per_method": {}, "paired_differences": {},
           "prediction_flips": {n: float(np.nanmean([pert_pred[i][f"pred_changed_{n}"] for i in audit_idx])) for n in SPEC_NAMES}}
for m in METHODS:
    sub = audit[audit.method == m]
    summary["per_method"][m] = {}
    for c in METRICS:
        v = sub[c]
        if v.notna().sum() == 0:
            continue
        summary["per_method"][m][c] = {
            "mean": float(v.mean()), "median": float(v.median()), "std": float(v.std()), "n": int(v.notna().sum()),
            "ci95_patient_bootstrap": cluster_bootstrap_ci(v.to_numpy(), sub.patient_id.to_numpy(), ACFG["n_bootstrap"], SEED),
        }

wide = {c: audit.pivot(index="row", columns="method", values=c) for c in METRICS}
pid = key.patient_id.to_numpy()
for a, b in [("gradcam", "integrated_gradients"), ("gradcam", "random"), ("integrated_gradients", "random")]:
    summary["paired_differences"][f"{a}_minus_{b}"] = {}
    for c in METRICS:
        if a not in wide[c] or b not in wide[c] or wide[c][[a, b]].notna().all(1).sum() == 0:
            continue
        diff = (wide[c][a] - wide[c][b]).to_numpy()
        summary["paired_differences"][f"{a}_minus_{b}"][c] = {
            "mean": float(np.nanmean(diff)),
            "ci95_patient_bootstrap": cluster_bootstrap_ci(diff, pid, ACFG["n_bootstrap"], SEED),
        }

with open(AUDIT_DIR / "audit_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

table = pd.DataFrame({m: {c: f"{d['mean']:.3f} [{d['ci95_patient_bootstrap'][0]:.3f}, {d['ci95_patient_bootstrap'][1]:.3f}]"
                          for c, d in summary["per_method"][m].items()} for m in METHODS})
print("mean [patient-bootstrap 95% CI]")
display(table)
print("paired differences (mean [95% CI]); an interval excluding 0 = difference survives patient resampling")
display(pd.DataFrame({k: {c: f"{d['mean']:+.3f} [{d['ci95_patient_bootstrap'][0]:+.3f}, {d['ci95_patient_bootstrap'][1]:+.3f}]"
                          for c, d in v.items()} for k, v in summary["paired_differences"].items()}))
print("fraction of audited images whose prediction flips under each perturbation:", summary["prediction_flips"])

## 11. Figures

One fixed colour per method throughout, so the figures read as a set: Grad-CAM, IG, and the
random control in grey. That way the control visually reads as "the floor".

In [ ]:
COLORS = {"gradcam": "#d95f02", "integrated_gradients": "#1b9e77", "random": "#9a9a9a"}
LABELS = {"gradcam": "Grad-CAM", "integrated_gradients": "Integrated Gradients", "random": "random control"}
xs = np.linspace(0, 1, STEPS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, mode in zip(axes, ["deletion", "insertion"]):
    for m in METHODS:
        c = curves[(m, mode)][audit_idx]
        med, q1, q3 = np.nanmedian(c, 0), np.nanpercentile(c, 25, 0), np.nanpercentile(c, 75, 0)
        auc_mean = np.nanmean([faith[(i, m)][f"{mode}_auc"] for i in audit_idx])
        ax.plot(xs, np.nanmean(c, 0), color=COLORS[m], lw=2, label=f"{LABELS[m]} (mean AUC {auc_mean:.3f})")
        ax.fill_between(xs, q1, q3, color=COLORS[m], alpha=0.15, lw=0)
    ax.set_title(f"{mode} ({'lower' if mode == 'deletion' else 'higher'} AUC = more faithful)")
    ax.set_xlabel("fraction of pixels " + ("removed" if mode == "deletion" else "restored"))
    ax.legend(fontsize=8, frameon=False)
axes[0].set_ylabel("p(originally predicted class)")
fig.suptitle("Faithfulness: mean curve, IQR band", y=1.02)
plt.tight_layout()
fig.savefig(FIG_DIR / "audit_deletion_insertion_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def boxplot_by_method(ax, col, methods=METHODS, title=None):
    data = [audit.loc[(audit.method == m), col].dropna().to_numpy() for m in methods]
    bp = ax.boxplot(data, patch_artist=True, widths=0.6, showfliers=False)
    for patch, m in zip(bp["boxes"], methods):
        patch.set_facecolor(COLORS[m]); patch.set_alpha(0.55)
    for k, (d, m) in enumerate(zip(data, methods), start=1):
        jitter = np.random.default_rng(k).uniform(-0.18, 0.18, len(d))
        ax.scatter(np.full(len(d), k) + jitter, d, s=3, color=COLORS[m], alpha=0.35)
    ax.set_xticks(range(1, len(methods) + 1), [LABELS[m] for m in methods], fontsize=8)
    ax.set_title(title or col, fontsize=10)


fig, axes = plt.subplots(1, 2, figsize=(11, 4))
boxplot_by_method(axes[0], "deletion_auc", title="deletion AUC (lower = better)")
boxplot_by_method(axes[1], "insertion_auc", title="insertion AUC (higher = better)")
plt.tight_layout()
fig.savefig(FIG_DIR / "audit_faithfulness_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, len(SPEC_NAMES), figsize=(4 * len(SPEC_NAMES), 7), sharey="row")
for col, name in enumerate(SPEC_NAMES):
    for r_, sim in enumerate(["spearman", "ssim"]):
        boxplot_by_method(axes[r_, col], f"{sim}_{name}", methods=EXPLAIN_METHODS, title=f"{sim} | {name}")
axes[0, 0].set_ylabel("Spearman (original vs aligned perturbed)")
axes[1, 0].set_ylabel("SSIM (original vs aligned perturbed)")
fig.suptitle("Stability: similarity of the explanation under label-preserving perturbations (1 = identical)", y=1.01)
plt.tight_layout()
fig.savefig(FIG_DIR / "audit_stability_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
boxplot_by_method(axes[0], "iou", title=f"IoU (heat >= {THR})")
boxplot_by_method(axes[1], "dice", title=f"Dice (heat >= {THR})")
boxplot_by_method(axes[2], "energy_ratio", title="energy in mask / mask area (1 = chance)")
axes[2].axhline(1, color="k", lw=0.8, ls="--")
hit = [audit.loc[audit.method == m, "pointing_hit"].mean() for m in METHODS]
axes[3].bar(range(len(METHODS)), hit, color=[COLORS[m] for m in METHODS], alpha=0.8)
axes[3].axhline(audit.loc[audit.method == "gradcam", "mask_frac"].mean(), color="k", lw=0.8, ls="--", label="chance (mean mask area)")
axes[3].set_xticks(range(len(METHODS)), [LABELS[m] for m in METHODS], fontsize=8)
axes[3].set_ylim(0, 1); axes[3].set_title("pointing game hit rate", fontsize=10); axes[3].legend(fontsize=8, frameon=False)
plt.tight_layout()
fig.savefig(FIG_DIR / "audit_localization_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

print("localization by true class (mean) -- pituitary tumors are small, so expect lower IoU there:")
display(audit[audit.method != "random"].groupby(["method", "true_name"])[["mask_frac", "iou", "dice", "pointing_hit", "energy_ratio"]]
        .mean().round(3))

### Grad-CAM vs IG, image by image

Each point is one test image. Above the diagonal, the y-axis method scored higher on that image.
Two things to look for. The first is *systematic* offset from the diagonal, i.e. one method is
better in general. The second is weak correlation between the methods: if the two disagree on
which images are well explained, then "how good is the explanation for *this* image" depends on
the method chosen, and that's a trust problem in itself. Misclassified images are marked with ×.

In [ ]:
pairs = [("deletion_auc", "deletion AUC (lower = better)"), ("insertion_auc", "insertion AUC (higher = better)"),
         ("energy_ratio", "energy ratio (higher = better)"), (f"spearman_{SPEC_NAMES[0]}", f"stability spearman, {SPEC_NAMES[0]}")]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.4))
wrong = ~audit.pivot(index="row", columns="method", values="correct")["gradcam"].astype(bool)
for ax, (c, title) in zip(axes, pairs):
    w = audit.pivot(index="row", columns="method", values=c)[["gradcam", "integrated_gradients"]].dropna()
    ok = ~wrong.loc[w.index]
    ax.scatter(w.loc[ok, "gradcam"], w.loc[ok, "integrated_gradients"], s=8, alpha=0.5, color="#555")
    ax.scatter(w.loc[~ok, "gradcam"], w.loc[~ok, "integrated_gradients"], s=30, marker="x", color="crimson", label="misclassified")
    lo, hi = np.nanmin(w.to_numpy()), np.nanmax(w.to_numpy())
    ax.plot([lo, hi], [lo, hi], "k--", lw=0.8)
    rho = AS.spearman_similarity(w.gradcam.to_numpy(), w.integrated_gradients.to_numpy()) if len(w) > 2 else float("nan")
    ax.set_title(f"{title}\nSpearman across images = {rho:.2f}", fontsize=9)
    ax.set_xlabel("Grad-CAM"); ax.set_ylabel("Integrated Gradients")
axes[0].legend(fontsize=8, frameon=False)
plt.tight_layout()
fig.savefig(FIG_DIR / "audit_gradcam_vs_ig.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Reload from disk and check

Everything downstream reads these files, not this notebook's memory, so the files themselves are
checked.

In [ ]:
back = pd.read_csv(AUDIT_DIR / "audit_results.csv")
assert len(back) == N * len(METHODS), (len(back), N, len(METHODS))
assert (back.row.to_numpy() == np.repeat(np.arange(N), len(METHODS))).all(), "row order broken"
assert (back.method.to_numpy() == np.tile(METHODS, N)).all()
for m in METHODS:
    assert (back.loc[back.method == m, "file_path"].to_numpy() == key.file_path.to_numpy()).all(), m
assert back.loc[back.in_model_audit & (back.method != "random"), STAB_COLS].notna().any(axis=1).all(), "missing stability values"
assert back.loc[back.in_model_audit, ["deletion_auc", "insertion_auc"]].notna().all().all()

cz = np.load(AUDIT_DIR / "audit_curves.npz")
assert all(cz[k].shape == (N, STEPS + 1) for k in cz.files if "__" in k)
with open(AUDIT_DIR / "audit_summary.json") as f:
    assert set(json.load(f)["per_method"]) == set(METHODS)

print(f"audit_results.csv: {len(back)} rows = {N} images x {len(METHODS)} methods, order matches test_predictions.csv")
print(back.groupby("method")[["deletion_auc", "insertion_auc", "iou", "pointing_hit", "energy_ratio"]].count())
!ls -la /kaggle/working/results/audit /kaggle/working/results/figures

## Interpreting the results

**What a trustworthy explainer looks like on each axis**

| axis | trustworthy | untrustworthy |
|---|---|---|
| faithfulness | deletion AUC clearly **below** random, insertion AUC clearly **above** random, with paired-difference CIs excluding 0 | about the same as the random control: the map ranks pixels no better than chance at predicting what the model relies on |
| stability | Spearman and SSIM close to 1 after alignment, for every perturbation, with a tight distribution | high median but a long lower tail (some images' explanations change completely under a 5° rotation), or a big gap between Spearman and SSIM |
| localization | energy ratio well above 1, pointing hit rate well above mask area, IoU/Dice clearly above random | at or near chance, or high only on large gliomas and collapsing on small pituitary tumors |

**How to read disagreements between axes**

- **Faithful but not localised:** the model really relies on pixels outside the tumor, e.g.
  surrounding oedema, skull shape, or acquisition artefacts. The explainer is doing its job, and
  what it reveals is a problem with the *model*. This is the most interesting kind of finding,
  and worth looking at image by image.
- **Localised but not faithful:** the map points at the tumor, yet deleting those pixels barely
  moves the prediction. The map looks right without saying anything about the model, which is
  exactly the failure that makes visual inspection alone insufficient.
- **Unstable:** if two scans that differ by a 5° rotation get different explanations, a clinician
  can't rely on either, however faithful the average is.

**What I expect for Grad-CAM vs IG** (these are hypotheses the numbers above can confirm or refute):

- **Localization:** Grad-CAM probably wins on energy and pointing for large tumors, since its
  smooth blobs cover regions well, but loses IoU on small pituitary tumors because of its 7×7
  resolution. IG's speckle is likely spread across edges of the whole brain, which lowers its
  energy ratio.
- **Stability:** Grad-CAM should be more stable. Its coarse maps barely change under a 5° rotation,
  while IG's pixel-level attributions are known to be sensitive to small input changes
  (Ghorbani et al., 2019). A low IG SSIM next to a moderate Spearman would match that pattern.
- **Faithfulness:** less clear. IG has a theoretical advantage (completeness), and deletion with a
  black baseline partly matches IG's own baseline, which favours IG somewhat. That's a known bias
  of deletion metrics and should be mentioned when comparing.
- **Caveats to report with the numbers:** ~35 test patients means wide CIs. Per-image
  normalisation affects threshold-based metrics. Deletion inserts off-distribution black patches,
  which partly measures how the model reacts to unusual inputs and not only faithfulness; that's
  why insertion (with a blurred start) is reported next to it.

## Closing checklist

- [ ] **Sanity tests passed** (§5). If they didn't, nothing below them counts.
- [ ] **Reproducibility check:** recomputed maps match notebook 2's (Spearman ≥ 0.99).
- [ ] Both explainers' deletion AUC is **below** the random control and insertion AUC **above** it.
  If either method isn't, that's a real result: report it, don't tune it away.
- [ ] `pred_changed_*` rates are low (a few % at most). If ±5° often flips the prediction, then the
  *model* is unstable, and the explanation-stability numbers need that context.
- [ ] `audit_results.csv` reloads with `N × 3` rows in `test_predictions.csv` order (asserted).
- [ ] If you ran with `subset_n`, set it back to `None` for the committed version reported in the
  write-up, or state the subset size next to every faithfulness/stability number.

**Save & Run All**
1. **Save Version → Save & Run All (Commit) → Save**, with the GPU on and all three inputs
   (notebooks 0, 1 and 2) still attached.
2. When it finishes, check the version's **Output** tab for `results/audit/` (`audit_results.csv`,
   `audit_summary.json`, `audit_curves.npz`, `audit_config.json`), the `audit_*.png` figures,
   `src/audit/` and `tests/`.
3. **Notebook 4 (MC-dropout uncertainty)** needs the checkpoint (notebook 1), the data (notebook 0)
   and `src/`. Attach **this** notebook for `src/` and the audit results, which notebook 5's joint
   analysis will merge with uncertainty by `file_path`. As before, put only the newest notebook's
   `src/` on `sys.path`.